
# Notebook 19 — Residual Phase Trajectories

Notebook 18 established that residual geometry contains predictive topology information.

Notebook 19 asks:

```text
How do topology classes move through residual manifold space as graph size increases?
```

Core claim:

```text
Residual topology identity is not static; each topology traces a scale-dependent path through residual manifold space.
```

Main outputs:

- topology trajectory PCA figure,
- trajectory length by topology,
- trajectory curvature,
- direction alignment matrix,
- radial distance vs graph size,
- phase drift vs leave-one-size-out accuracy,
- trajectory metrics CSV,
- trajectory summary JSON.


## Imports and setup

In [ ]:
import json
import zipfile
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)

# ---------------------------------------------------
# robust repo detection for Colab + local Jupyter
# ---------------------------------------------------

def detect_repo_root():
    cwd = Path.cwd()

    # case 1: already inside repo root
    if (cwd / "results").exists() and (cwd / "notebooks").exists():
        return cwd

    # case 2: running inside notebooks/
    if cwd.name == "notebooks":
        parent = cwd.parent
        if (parent / "results").exists():
            return parent

    # case 3: search entire /content tree for expected files
    search_root = Path("/content") if Path("/content").exists() else cwd

    matches = list(
        search_root.rglob("residual_classification_feature_matrix.csv")
    )

    if len(matches) > 0:
        # file is inside repo/results/
        return matches[0].parents[1]

    matches = list(
        search_root.rglob("residual_geometry_features.csv")
    )

    if len(matches) > 0:
        return matches[0].parents[1]

    # fallback
    return cwd

REPO_ROOT = detect_repo_root()

RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"

FIG_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

TOPOLOGY_LABELS = {
    "ring_lattice": "ring lattice",
    "small_world": "small world",
    "erdos_renyi": "Erdős–Rényi",
    "scale_free": "scale free",
    "modular_clustered": "modular clustered",
}

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("results dir:", RESULTS_DIR)

print("\nresults contents:")
if RESULTS_DIR.exists():
    for p in sorted(RESULTS_DIR.glob("*")):
        print(" -", p.name)
else:
    print("results dir missing")

print("\nReady.")

## Load residual geometry data

In [ ]:
feature_path = RESULTS_DIR / "residual_classification_feature_matrix.csv"
geometry_path = RESULTS_DIR / "residual_geometry_features.csv"
embedding_path = RESULTS_DIR / "residual_pca_embedding.csv"
loso_path = RESULTS_DIR / "residual_leave_one_size_out_predictions.csv"

print("checking feature_path:", feature_path)
print("exists:", feature_path.exists())
print("checking geometry_path:", geometry_path)
print("exists:", geometry_path.exists())

if feature_path.exists():
    feature_df = pd.read_csv(feature_path)
    data_source = "loaded Notebook 18 feature matrix"

elif geometry_path.exists():
    geometry_df = pd.read_csv(geometry_path)
    geometry_df["label"] = geometry_df["topology"].map(TOPOLOGY_LABELS)
    feature_df = geometry_df.copy()
    data_source = "loaded Notebook 17 geometry features"

else:
    available = sorted([p.name for p in RESULTS_DIR.glob("*")]) if RESULTS_DIR.exists() else []
    raise FileNotFoundError(
        "Missing residual feature inputs.\n\n"
        f"cwd: {Path.cwd()}\n"
        f"repo root: {REPO_ROOT}\n"
        f"results dir: {RESULTS_DIR}\n\n"
        f"Expected one of:\n"
        f"- {feature_path}\n"
        f"- {geometry_path}\n\n"
        f"Available files in RESULTS_DIR:\n{available}"
    )

feature_df = feature_df.replace([np.inf, -np.inf], np.nan).dropna()
feature_df["label"] = feature_df["topology"].map(TOPOLOGY_LABELS)
feature_df = feature_df.sort_values(["topology", "n_modules"]).reset_index(drop=True)

print("data source:", data_source)
print("shape:", feature_df.shape)
feature_df.head()

## Recompute a shared PCA manifold

In [ ]:

CANDIDATE_FEATURES = [
    "mean_abs_residual",
    "max_abs_residual",
    "total_residual_energy",
    "residual_localization",
    "residual_asymmetry",
    "residual_entropy",
    "residual_bend_energy",
    "mean_abs_bend",
    "residual_spectral_ratio",
]

feature_cols = [c for c in CANDIDATE_FEATURES if c in feature_df.columns]

X = feature_df[feature_cols].to_numpy(dtype=float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
Z = pca.fit_transform(X_scaled)

traj_df = feature_df[["topology", "label", "n_modules"]].copy()
traj_df["pc1"] = Z[:, 0]
traj_df["pc2"] = Z[:, 1]

traj_df.to_csv(RESULTS_DIR / "residual_phase_trajectory_embedding.csv", index=False)

print("features:", feature_cols)
print("PCA explained variance:", pca.explained_variance_ratio_)
traj_df.head()


## Figure 1 — Topology trajectory paths

In [ ]:

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    if sub.empty:
        continue

    plt.plot(
        sub["pc1"],
        sub["pc2"],
        marker="o",
        linewidth=2,
        markersize=8,
        label=TOPOLOGY_LABELS[topology],
    )

    # arrows between graph sizes
    for i in range(len(sub) - 1):
        x0, y0 = sub.iloc[i][["pc1", "pc2"]]
        x1, y1 = sub.iloc[i + 1][["pc1", "pc2"]]
        plt.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.7),
        )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["pc1"], row["pc2"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            alpha=0.8,
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title("Residual phase trajectories across graph size")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "topology_trajectory_pca.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Trajectory metrics

In [ ]:

def path_length(points):
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return 0.0
    return float(np.sum(np.linalg.norm(np.diff(points, axis=0), axis=1)))

def path_curvature(points):
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        return 0.0
    second = points[2:] - 2 * points[1:-1] + points[:-2]
    return float(np.sum(np.linalg.norm(second, axis=1)))

def endpoint_vector(points):
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return np.zeros(2)
    return points[-1] - points[0]

trajectory_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    points = sub[["pc1", "pc2"]].to_numpy(dtype=float)

    L = path_length(points)
    K = path_curvature(points)
    v = endpoint_vector(points)
    straight = float(np.linalg.norm(v))
    tortuosity = float(L / max(straight, 1e-9))

    trajectory_rows.append({
        "topology": topology,
        "label": TOPOLOGY_LABELS[topology],
        "n_points": int(len(points)),
        "trajectory_length": L,
        "endpoint_distance": straight,
        "trajectory_curvature": K,
        "trajectory_tortuosity": tortuosity,
        "endpoint_dx": float(v[0]),
        "endpoint_dy": float(v[1]),
    })

trajectory_metrics_df = pd.DataFrame(trajectory_rows)
trajectory_metrics_df.to_csv(RESULTS_DIR / "residual_phase_trajectory_metrics.csv", index=False)

trajectory_metrics_df


## Figure 2 — Trajectory length by topology

In [ ]:

plot_df = trajectory_metrics_df.sort_values("trajectory_length", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["label"], plot_df["trajectory_length"])
plt.xlabel("PCA trajectory length")
plt.title("Residual manifold trajectory length by topology")
plt.grid(alpha=0.3, axis="x")

fig_path = FIG_DIR / "trajectory_length_by_topology.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Figure 3 — Trajectory curvature

In [ ]:

plot_df = trajectory_metrics_df.sort_values("trajectory_curvature", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["label"], plot_df["trajectory_curvature"])
plt.xlabel("trajectory curvature")
plt.title("Residual trajectory curvature by topology")
plt.grid(alpha=0.3, axis="x")

fig_path = FIG_DIR / "trajectory_curvature.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Figure 4 — Direction alignment matrix

In [ ]:

vectors = {}
for topology in TOPOLOGIES:
    row = trajectory_metrics_df[trajectory_metrics_df["topology"] == topology].iloc[0]
    vectors[topology] = np.array([row["endpoint_dx"], row["endpoint_dy"]], dtype=float)

alignment = np.zeros((len(TOPOLOGIES), len(TOPOLOGIES)))

for i, t1 in enumerate(TOPOLOGIES):
    for j, t2 in enumerate(TOPOLOGIES):
        u = vectors[t1]
        v = vectors[t2]
        denom = np.linalg.norm(u) * np.linalg.norm(v)
        alignment[i, j] = float(np.dot(u, v) / denom) if denom > 0 else np.nan

alignment_df = pd.DataFrame(
    alignment,
    index=[TOPOLOGY_LABELS[t] for t in TOPOLOGIES],
    columns=[TOPOLOGY_LABELS[t] for t in TOPOLOGIES],
)

alignment_out = alignment_df.copy()
alignment_out.insert(0, "topology", alignment_out.index)
alignment_out.to_csv(RESULTS_DIR / "residual_direction_alignment.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(alignment, vmin=-1, vmax=1)

ax.set_xticks(range(len(TOPOLOGIES)))
ax.set_yticks(range(len(TOPOLOGIES)))
ax.set_xticklabels([TOPOLOGY_LABELS[t] for t in TOPOLOGIES], rotation=45, ha="right")
ax.set_yticklabels([TOPOLOGY_LABELS[t] for t in TOPOLOGIES])

for i in range(len(TOPOLOGIES)):
    for j in range(len(TOPOLOGIES)):
        ax.text(j, i, f"{alignment[i, j]:.2f}", ha="center", va="center")

ax.set_title("Residual trajectory direction alignment")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="cosine alignment")
plt.tight_layout()

fig_path = FIG_DIR / "direction_alignment_matrix.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

alignment_df


## Figure 5 — Radial distance vs graph size

In [ ]:

center = traj_df[["pc1", "pc2"]].mean().to_numpy(dtype=float)

radial_df = traj_df.copy()
radial_df["radial_distance"] = np.linalg.norm(
    radial_df[["pc1", "pc2"]].to_numpy(dtype=float) - center,
    axis=1,
)

radial_df.to_csv(RESULTS_DIR / "residual_radial_growth.csv", index=False)

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = radial_df[radial_df["topology"] == topology].sort_values("n_modules")
    plt.plot(
        sub["n_modules"],
        sub["radial_distance"],
        marker="o",
        linewidth=2,
        label=TOPOLOGY_LABELS[topology],
    )

plt.xlabel("graph size N")
plt.ylabel("distance from global residual centroid")
plt.title("Residual radial distance vs graph size")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "radial_distance_vs_size.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

radial_df.head()


## Figure 6 — Phase drift vs classification stability

In [ ]:

if loso_path.exists():
    loso_predictions = pd.read_csv(loso_path)

    stability_rows = []
    for topology in TOPOLOGIES:
        sub = loso_predictions[loso_predictions["topology"] == topology]
        if "loso_correct" in sub.columns:
            acc = float(sub["loso_correct"].mean())
        else:
            acc = np.nan

        traj = trajectory_metrics_df[trajectory_metrics_df["topology"] == topology].iloc[0]

        stability_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "trajectory_length": float(traj["trajectory_length"]),
            "trajectory_curvature": float(traj["trajectory_curvature"]),
            "leave_one_size_out_accuracy": acc,
        })

    stability_df = pd.DataFrame(stability_rows)
    stability_note = "loaded Notebook 18 leave-one-size-out predictions"
else:
    stability_df = trajectory_metrics_df[[
        "topology", "label", "trajectory_length", "trajectory_curvature"
    ]].copy()
    stability_df["leave_one_size_out_accuracy"] = np.nan
    stability_note = "Notebook 18 leave-one-size-out predictions unavailable"

stability_df.to_csv(RESULTS_DIR / "residual_phase_drift_vs_accuracy.csv", index=False)

plt.figure(figsize=(8, 6))

valid = stability_df.dropna(subset=["leave_one_size_out_accuracy"])
if len(valid) > 0:
    plt.scatter(
        valid["trajectory_length"],
        valid["leave_one_size_out_accuracy"],
        s=140,
        alpha=0.75,
    )

    for _, row in valid.iterrows():
        plt.annotate(
            row["label"],
            xy=(row["trajectory_length"], row["leave_one_size_out_accuracy"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=9,
        )

    plt.xlabel("residual trajectory length")
    plt.ylabel("leave-one-size-out accuracy")
    plt.title("Phase drift vs topology classification stability")
    plt.ylim(-0.05, 1.05)
else:
    plt.text(
        0.5, 0.5,
        "Notebook 18 leave-one-size-out predictions unavailable",
        ha="center",
        va="center",
        transform=plt.gca().transAxes,
    )
    plt.xlabel("residual trajectory length")
    plt.ylabel("leave-one-size-out accuracy")
    plt.title("Phase drift vs topology classification stability")

plt.grid(alpha=0.3)

fig_path = FIG_DIR / "phase_drift_vs_accuracy.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

print(stability_note)
stability_df


## Centroid flow and scale drift

In [ ]:

centroid_rows = []

for N, subN in traj_df.groupby("n_modules"):
    center_N = subN[["pc1", "pc2"]].mean().to_numpy(dtype=float)

    for topology in TOPOLOGIES:
        sub = subN[subN["topology"] == topology]
        if sub.empty:
            continue

        p = sub[["pc1", "pc2"]].iloc[0].to_numpy(dtype=float)
        centroid_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "n_modules": int(N),
            "pc1": float(p[0]),
            "pc2": float(p[1]),
            "centered_pc1": float(p[0] - center_N[0]),
            "centered_pc2": float(p[1] - center_N[1]),
            "distance_from_size_centroid": float(np.linalg.norm(p - center_N)),
        })

centroid_flow_df = pd.DataFrame(centroid_rows)
centroid_flow_df.to_csv(RESULTS_DIR / "residual_centroid_flow.csv", index=False)

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = centroid_flow_df[centroid_flow_df["topology"] == topology].sort_values("n_modules")
    plt.plot(
        sub["centered_pc1"],
        sub["centered_pc2"],
        marker="o",
        linewidth=2,
        label=TOPOLOGY_LABELS[topology],
    )

    for i in range(len(sub) - 1):
        x0, y0 = sub.iloc[i][["centered_pc1", "centered_pc2"]]
        x1, y1 = sub.iloc[i + 1][["centered_pc1", "centered_pc2"]]
        plt.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.7),
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel("PC1 relative to size centroid")
plt.ylabel("PC2 relative to size centroid")
plt.title("Residual centroid flow after size-centering")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "topology_centroid_flow.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

centroid_flow_df.head()


## Compact trajectory table

In [ ]:

display_cols = [
    "label",
    "trajectory_length",
    "endpoint_distance",
    "trajectory_curvature",
    "trajectory_tortuosity",
]

trajectory_metrics_df[display_cols].sort_values("trajectory_length", ascending=False)


## Summary export

In [ ]:

max_length_row = trajectory_metrics_df.sort_values("trajectory_length", ascending=False).iloc[0]
min_length_row = trajectory_metrics_df.sort_values("trajectory_length", ascending=True).iloc[0]
max_curv_row = trajectory_metrics_df.sort_values("trajectory_curvature", ascending=False).iloc[0]

summary = {
    "notebook": "19_residual_phase_trajectories.ipynb",
    "core_question": "How do topology classes move through residual manifold space as graph size increases?",
    "core_claim": "Residual topology identity is not static; each topology traces a scale-dependent path through residual manifold space.",
    "data_source": data_source,
    "features": feature_cols,
    "pca_explained_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
    "largest_trajectory_length": {
        "topology": str(max_length_row["topology"]),
        "label": str(max_length_row["label"]),
        "value": float(max_length_row["trajectory_length"]),
    },
    "smallest_trajectory_length": {
        "topology": str(min_length_row["topology"]),
        "label": str(min_length_row["label"]),
        "value": float(min_length_row["trajectory_length"]),
    },
    "largest_trajectory_curvature": {
        "topology": str(max_curv_row["topology"]),
        "label": str(max_curv_row["label"]),
        "value": float(max_curv_row["trajectory_curvature"]),
    },
    "figures": [
        "topology_trajectory_pca.png",
        "trajectory_length_by_topology.png",
        "trajectory_curvature.png",
        "direction_alignment_matrix.png",
        "radial_distance_vs_size.png",
        "phase_drift_vs_accuracy.png",
        "topology_centroid_flow.png",
    ],
    "results": [
        "residual_phase_trajectory_embedding.csv",
        "residual_phase_trajectory_metrics.csv",
        "residual_direction_alignment.csv",
        "residual_radial_growth.csv",
        "residual_phase_drift_vs_accuracy.csv",
        "residual_centroid_flow.csv",
        "residual_phase_trajectory_summary.json",
    ],
}

summary_path = RESULTS_DIR / "residual_phase_trajectory_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 19 — Residual Phase Trajectories",
    "",
    "**Core question:** How do topology classes move through residual manifold space as graph size increases?",
    "",
    "**Core claim:** Residual topology identity is not static; each topology traces a scale-dependent path through residual manifold space.",
    "",
    "Recommended paper figures:",
    "",
    "- `figures/topology_trajectory_pca.png`",
    "- `figures/trajectory_length_by_topology.png`",
    "- `figures/direction_alignment_matrix.png`",
    "- `figures/topology_centroid_flow.png`",
    "",
    "Key interpretation:",
    "",
    "Residual topology identities behave like geometric flow trajectories through a residual manifold.",
    "",
]

doc_path = DOCS_DIR / "notebook_19_residual_phase_trajectories.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")



## Interpretation

Notebook 17:

```text
residual geometry exists
```

Notebook 18:

```text
residual geometry classifies topology
```

Notebook 19:

```text
topology evolves through residual manifold trajectories
```

This supports the paper-level framing:

```text
Collapse exposes a residual manifold, and graph topology moves through that manifold as scale changes.
```


## Optional export zip

In [ ]:

zip_path = REPO_ROOT / "notebook_19_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
